In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Запрашиваем модель и параметры
MODEL_ID = "intfloat/multilingual-e5-large"  # ID модели для встраивания
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)  # Загружаем токенизатор
model = AutoModel.from_pretrained(MODEL_ID)  # Загружаем модель

# Пример документов для индексации
passages = [
    {
        "id": "doc1",
        "language": "en",
        "passage": "I sat on the bank of the river today."
    },
    {
        "id": "doc2",
        "language": "de",
        "passage": "Ich bin heute zum Flussufer gegangen."
    },
    {
        "id": "doc3",
        "language": "en",
        "passage": "I walked to the bank today to deposit money."
    },
    {
        "id": "doc4",
        "language": "de",
        "passage": "Ich saß heute bei der Bank und wartete auf mein Geld."
    },
]

# Функция для получения встраиваний текста
def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**inputs).last_hidden_state.mean(dim=1)  # Получаем среднее встраивание
    return embeddings.numpy()

# Индексируем документы и получаем их встраивания
indexed_passages = []
for doc in passages:
    embedding = get_embedding(f"passage: {doc['passage']}")  # Получаем встраивание с префиксом
    indexed_passages.append({
        "id": doc["id"],
        "language": doc["language"],
        "passage": doc["passage"],
        "embedding": embedding
    })

def query(q):
    """Функция для выполнения запроса с использованием встраиваний."""
    query_embedding = get_embedding(f"query: {q}")  # Получаем встраивание для запроса
    similarities = []

    # Вычисляем косинусное сходство между запросом и индексированными документами
    for doc in indexed_passages:
        score = cosine_similarity(query_embedding, doc['embedding'])[0][0]
        similarities.append((score, doc))

    # Сортируем по схожести и возвращаем топ-4 результата
    similarities.sort(key=lambda x: x[0], reverse=True)
    return similarities[:4]

def pretty_response(results):
    """Функция для красивого вывода результатов поиска."""
    for score, doc in results:
        print()
        print(f"ID: {doc['id']}")
        print(f"Язык: {doc['language']}")
        print(f"Текст: {doc['passage']}")
        print(f"Оценка: {score:.4f}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Выполняем поиск по запросу и выводим результаты


In [ ]:
pretty_response(query("riverside"))


ID: doc1
Язык: en
Текст: I sat on the bank of the river today.
Оценка: 0.7746

ID: doc2
Язык: de
Текст: Ich bin heute zum Flussufer gegangen.
Оценка: 0.7484

ID: doc4
Язык: de
Текст: Ich saß heute bei der Bank und wartete auf mein Geld.
Оценка: 0.7024

ID: doc3
Язык: en
Текст: I walked to the bank today to deposit money.
Оценка: 0.7012


In [ ]:
pretty_response(query("Geldautomat"))


ID: doc4
Язык: de
Текст: Ich saß heute bei der Bank und wartete auf mein Geld.
Оценка: 0.7972

ID: doc3
Язык: en
Текст: I walked to the bank today to deposit money.
Оценка: 0.7580

ID: doc2
Язык: de
Текст: Ich bin heute zum Flussufer gegangen.
Оценка: 0.7162

ID: doc1
Язык: en
Текст: I sat on the bank of the river today.
Оценка: 0.7053


In [ ]:
pretty_response(query("movement"))


ID: doc3
Язык: en
Текст: I walked to the bank today to deposit money.
Оценка: 0.7493

ID: doc2
Язык: de
Текст: Ich bin heute zum Flussufer gegangen.
Оценка: 0.7290

ID: doc1
Язык: en
Текст: I sat on the bank of the river today.
Оценка: 0.7226

ID: doc4
Язык: de
Текст: Ich saß heute bei der Bank und wartete auf mein Geld.
Оценка: 0.7188


In [ ]:
pretty_response(query("stillness"))


ID: doc1
Язык: en
Текст: I sat on the bank of the river today.
Оценка: 0.7286

ID: doc4
Язык: de
Текст: Ich saß heute bei der Bank und wartete auf mein Geld.
Оценка: 0.7268

ID: doc2
Язык: de
Текст: Ich bin heute zum Flussufer gegangen.
Оценка: 0.7079

ID: doc3
Язык: en
Текст: I walked to the bank today to deposit money.
Оценка: 0.7032
